In [8]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("UrgenceNet - Partie 2 - RDD")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4")
    .config("spark.jars.ivy", "/home/spark/.ivy2")
    .config("spark.hadoop.fs.s3a.endpoint", "minio:9000")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.access.key", "urgenceadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "urgence-local-2026")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)

sc = spark.sparkContext
print("Spark :", spark.version)
print("Master :", sc.master)

Spark : 3.5.7
Master : spark://spark-master:7077


In [9]:
import json

history_path = "s3a://lake/bronze/history/urgences_history.jsonl"

raw_lines = sc.textFile(history_path)
events = raw_lines.map(json.loads).cache()

event_count = events.count()
print("Événements lus :", event_count)
print("Exemple :", events.first())

Événements lus : 50000
Exemple : {'visit_id': 'V-0004877', 'patient_id': 'P-05811', 'facility_id': 'F-011', 'city': 'Rabat', 'department': 'Traumatologie', 'status': 'ARRIVED', 'priority': None, 'wait_minutes': None, 'desk_id': None, 'transfer_to': None, 'event_time': '2026-11-15T00:01:13Z'}


In [11]:
care_started = events.filter(lambda event: event["status"] == "CARE_STARTED")

city_totals = care_started.map(
    lambda event: (event["city"], (event["wait_minutes"], 1))
).reduceByKey(
    lambda left, right: (left[0] + right[0], left[1] + right[1])
)

sc.setJobGroup("rdd-city-stats", "RDD - prises en charge et délai moyen par ville")

try:
    city_results = sorted(city_totals.collect())
finally:
    sc.setLocalProperty("spark.jobGroup.id", None)
    sc.setLocalProperty("spark.job.description", None)
    sc.setLocalProperty("spark.job.interruptOnCancel", None)

print(f"{'Ville':<14} {'Prises en charge':>17} {'Attente moyenne (min)':>23}")

for city, (total_wait, count) in city_results:
    print(f"{city:<14} {count:>17} {total_wait / count:>23.2f}")

[Stage 10:>                                                         (0 + 2) / 2]

Ville           Prises en charge   Attente moyenne (min)
Casablanca                  3079                   47.61
Marrakech                   3194                   47.25
Rabat                       3144                   47.30
Tanger                      3122                   47.94


In [12]:
triaged = events.filter(lambda event: event["status"] == "TRIAGED")

priority_counts = triaged.map(
    lambda event: (event["priority"], 1)
).reduceByKey(
    lambda left, right: left + right
)

print("Passages triés par priorité")
for priority, count in sorted(priority_counts.collect()):
    print(f"Priorité {priority} : {count}")

Passages triés par priorité
Priorité 1 : 2498
Priorité 2 : 2565
Priorité 3 : 2459
Priorité 4 : 2490
Priorité 5 : 2482


In [13]:
facility_counts = care_started.map(
    lambda event: (event["facility_id"], 1)
).reduceByKey(
    lambda left, right: left + right
)

top_5 = facility_counts.takeOrdered(
    5, key=lambda row: (-row[1], row[0])
)

print("Top 5 des établissements par prises en charge")
for rank, (facility_id, count) in enumerate(top_5, start=1):
    print(f"{rank}. {facility_id} : {count} prises en charge")

Top 5 des établissements par prises en charge
1. F-028 : 339 prises en charge
2. F-027 : 334 prises en charge
3. F-030 : 334 prises en charge
4. F-024 : 332 prises en charge
5. F-012 : 330 prises en charge


In [14]:
print(city_totals.toDebugString().decode("utf-8"))

(2) PythonRDD[29] at collect at /tmp/ipykernel_272/482302101.py:12 []
 |  MapPartitionsRDD[28] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[27] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[26] at reduceByKey at /tmp/ipykernel_272/482302101.py:5 []
    |  PythonRDD[25] at reduceByKey at /tmp/ipykernel_272/482302101.py:5 []
    |  PythonRDD[17] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 2; MemorySize: 1767.3 KiB; DiskSize: 0.0 B
    |  s3a://lake/bronze/history/urgences_history.jsonl MapPartitionsRDD[16] at textFile at NativeMethodAccessorImpl.java:0 []
    |  s3a://lake/bronze/history/urgences_history.jsonl HadoopRDD[15] at textFile at NativeMethodAccessorImpl.java:0 []


26/11/19 15:47:55 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/blockmgr-49df2ac5-8591-4535-9314-644cdd41b7f1. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/blockmgr-49df2ac5-8591-4535-9314-644cdd41b7f1
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:174)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1(DiskBlockManager.scala:368)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1$adapted(DiskBlockManager.scala:364)
	at scala.collection.IndexedSeqOptimize